# 03. Biểu diễn BERT (B0–B5) và A3 (TV3)

Tính 6 embedding, lưu vào `embeddings/`, chạy A3 (BERT + K-Means, k = 6), ghi `results.csv` và cột `bert_kmeans` của `doc_topics.csv`.

Notebook chỉ gọi hàm trong `src/`. Embedding đã có file thì **không tính lại**.

In [ ]:
import sys; sys.path.append("..")
from src.config import *

In [ ]:
import time
import numpy as np
import pandas as pd
from IPython.display import display

from src.preprocess import load_articles, get_tokens
from src.embed import compute_all, load_embedding, embed_query, run_bert_kmeans
from src.baselines import top_words_by_cluster
from src.metrics import evaluate, make_row, save_results

## 1. Dữ liệu

In [ ]:
df = load_articles()
print(df.shape)
print(df["true_label"].value_counts().to_string())

## 2. Tính embedding B0–B5

Mỗi khóa tính một lần và lưu `embeddings/<khóa>.npy`; khóa đã có file sẽ được bỏ qua. Muốn tính lại một khóa: `compute_all(df, keys=["b3_minilm"], overwrite=True)`.

Lần đầu mỗi mô hình sẽ tải từ Hugging Face (cần mạng). Thời gian tải **không** được tính vào thời gian embedding.

In [ ]:
time_table = compute_all(df)
display(time_table)

## 3. Kiểm tra embedding

In [ ]:
for key in EMBEDDINGS:
    X = load_embedding(key)                      # tự kiểm tra số dòng bằng số bài
    norms = np.linalg.norm(X, axis=1)
    print(f"{key:15s} shape={X.shape} dtype={X.dtype} NaN={np.isnan(X).any()} "
          f"norm trong [{norms.min():.3f}, {norms.max():.3f}]")
    assert X.shape[0] == len(df) and not np.isnan(X).any() and np.allclose(norms, 1, atol=1e-3)
print("Các file embedding hợp lệ.")

Kiểm tra `embed_query` (app demo sẽ dùng): vector của một bài tính lại từ văn bản phải gần như trùng vector đã lưu (cosine ≈ 1).

In [ ]:
i = 0
for key in EMBEDDINGS:
    if EMBEDDINGS[key]["model"] is None:
        continue                                 # B0 không nhúng được văn bản mới
    q = embed_query(df.loc[i, "text_bert"], key)[0]
    cos = float(q @ load_embedding(key)[i])
    print(f"{key:15s} cosine(embed_query, đã lưu) = {cos:.4f}", "" if cos > 0.99 else "  <-- LỆCH, kiểm tra embed_query")

## 4. A3: BERT + K-Means (k = 6)

`time_s` = thời gian embedding + thời gian K-Means (để so sánh được với LSA, vốn gồm cả bước biểu diễn).

In [ ]:
emb_time = pd.read_csv(EMB_TIME_CSV).set_index("embedding")["time_s"]
tokens = get_tokens(df)
text_tok = df["text_tok"].tolist()
y_true = df["true_label"].values


def chay_a3(key, seed):
    X = load_embedding(key)
    t0 = time.perf_counter()
    labels = run_bert_kmeans(X, k=N_TOPICS, seed=seed)
    t_km = time.perf_counter() - t0
    words = top_words_by_cluster(text_tok, labels)
    scores = evaluate(labels, y_true, words, tokens)
    row = make_row("bert_kmeans", key, "k6", "", seed, scores, float(emb_time[key]) + t_km)
    return labels, words, row

In [ ]:
rows = [chay_a3(key, SEED)[2] for key in EMBEDDINGS]
save_results(rows)
df_a3 = (pd.DataFrame(rows)[["embedding", "nmi", "ari", "purity", "cv", "diversity", "time_s"]]
         .sort_values("nmi", ascending=False).reset_index(drop=True))
display(df_a3)

## 5. Chọn embedding BERT tốt nhất (NMI cao nhất, k = 6) và chạy 3 seed

B0 không dùng BERT nên không tham gia chọn cho A3.

In [ ]:
bert_keys = [k for k in EMBEDDINGS if k != "b0_tfidf_svd"]
best = df_a3[df_a3["embedding"].isin(bert_keys)].iloc[0]["embedding"]
print("Embedding BERT tốt nhất (NMI, k=6):", best, "| BEST_EMBEDDING hiện tại trong config:", BEST_EMBEDDING)

rows_seed, labels_best = [], None
for s in SEEDS:
    labels, words, row = chay_a3(best, s)
    rows_seed.append(row)
    if s == SEED:
        labels_best, words_best = labels, words
save_results(rows_seed)

df_seed = pd.DataFrame(rows_seed)
summary = df_seed[["ari", "nmi", "purity", "cv", "diversity"]].agg(["mean", "std"]).round(4)
display(summary)

## 6. Ghi nhãn vào `doc_topics.csv`

Chỉ thêm/cập nhật cột `bert_kmeans` (nhãn của cấu hình tốt nhất, seed 42). Nếu file đã có thì giữ nguyên các cột của người khác.

In [ ]:
labels_str = pd.Series(labels_best).astype(int).astype(str).values
if DOC_TOPICS_CSV.exists():
    dt = pd.read_csv(DOC_TOPICS_CSV, dtype=str, keep_default_na=False)   # giữ nguyên chữ, không đổi kiểu cột người khác
    assert (dt["id"].values == df["id"].values).all(), "doc_topics.csv khác thứ tự với articles.csv"
else:
    dt = pd.DataFrame({"id": df["id"], "true_label": df["true_label"]})
dt["bert_kmeans"] = labels_str
TABLES_DIR.mkdir(parents=True, exist_ok=True)
dt.to_csv(DOC_TOPICS_CSV, index=False, encoding="utf-8")
print("Đã ghi cột bert_kmeans vào", DOC_TOPICS_CSV.name, "| các cột:", list(dt.columns))

## 7. Xem chủ đề tìm được (cấu hình tốt nhất)

In [ ]:
for t, w in enumerate(words_best):
    print(f"Chủ đề {t}: {' '.join(w)}")
display(pd.crosstab(pd.Series(labels_best, name="cụm"), pd.Series(y_true, name="nhãn thật")))

## 8. Bốn phép so sánh trọng tâm (điền nhận xét vào `ket_qua/03_bieu_dien_bert.md`)

In [ ]:
nmi = df_a3.set_index("embedding")["nmi"]
print(f"B1 [CLS] {nmi['b1_mbert_cls']:.4f}  vs  B2 mean {nmi['b2_mbert_mean']:.4f}")
print(f"mBERT nguyên bản (tốt nhất B1/B2) {max(nmi['b1_mbert_cls'], nmi['b2_mbert_mean']):.4f}"
      f"  vs  SBERT (tốt nhất B3–B5) {nmi[['b3_minilm', 'b4_vi_sbert', 'b5_e5']].max():.4f}")
print(f"B3 đa ngôn ngữ {nmi['b3_minilm']:.4f}  vs  B4 tiếng Việt {nmi['b4_vi_sbert']:.4f}")
print(f"B0 tần suất {nmi['b0_tfidf_svd']:.4f}  vs  BERT tốt nhất {nmi[bert_keys].max():.4f}")
print("\nBáo TV4 và TV5 cập nhật BEST_EMBEDDING trong config thành:", best)